# Notebook overview
Import the libraries needed to assemble machine-generated annotations into Caesar upload tables and then import those tables into Caesar.

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import string
import random
import json
import os
import glob
import getpass

from panoptes_client import Panoptes, Caesar, Workflow


## Part 1: Uploading data for correct-a-box (Clump Scout II)

### Step 1: Ensure subject manifests already exist
The subject-upload step is now handled by the upload-subjects notebook, which creates the subject manifests consumed here. This notebook starts from those existing manifest CSV files rather than uploading subjects itself.

### Step 2: Run the cells in Part 1 of this notebook
This will generate a CSV file to be imported into the Caesar database.

The machine learning predictions file was provided by Jürgen Popp.

The `subject_manifest.csv` files are generated by the upload-subjects workflow and provide the subject-to-filename mapping used here.

All these CSV files are expected to be stored in the data directories configured below.

### Step 3: Import annotation data into Caesar database
First, upload the `data_file_to_caesar` to Google drive and generate a public sharing link. Then, update the cell in the Part 2 of this notebook to use the unique file ID that is embedded in that sharing link and run that cell. If all goes well, it should return `(None, None)`.


## Part 1: Generate CSV for import into Caesar database

In [ ]:
DATA_DIR = "/Users/hjd229/Documents/Data/Clump_Scout_2"
IMAGE_DIR = "subject_images"
MANIFEST_DIR = "subject_manifests"
ANNOTATIONS_DIR = "subject_annotations"
PREDICTIONS_DIR = "FRCNN_predictions"
PREDICTIONS_FILE = "preds_Euclid_nms.gzip"

## Load detections and manifests
Read the machine-predicted bounding boxes, pre-group them by object and label, and collect every subject manifest that will be converted into Caesar-ready annotation rows.

In [ ]:
bboxes = pd.read_parquet(os.path.join(DATA_DIR, PREDICTIONS_DIR, PREDICTIONS_FILE))
bbox_groups = bboxes.groupby(["local_ids", "labels"])

subject_manifests = glob.glob(os.path.join(DATA_DIR, MANIFEST_DIR, "*.csv"))

Count occurrences of each label to intuit/verify which category each pertains to. The nominal mapping between tools and categories is hard-coded below but it may need updating if the model changes in the future.

In [22]:
bboxes.labels.value_counts()

labels
1    2892943
4     895439
3     154664
5     101120
2        767
Name: count, dtype: int64

Display tools in the workflow task. Again, this is to ensure correct label-to-tool mapping.

In [24]:
wf = Workflow.find(29070)
print(wf.display_name)
print(json.dumps(wf.raw["tasks"]["T0"]["tools"], indent="\t"))

Correct a clump
[
	{
		"type": "rectangle",
		"color": "#00ff00",
		"label": "Clumps",
		"details": []
	},
	{
		"type": "rectangle",
		"color": "#ffff00",
		"label": "Foreground stars",
		"details": []
	},
	{
		"type": "rectangle",
		"color": "#0000ff",
		"label": "Other galaxies",
		"details": []
	},
	{
		"type": "rectangle",
		"color": "#ff0000",
		"label": "Galaxy bulge",
		"details": []
	}
]


Some data processing steps that only need to be done once.

In [25]:

unique_labels = bboxes.labels.unique()
extractor_key = "machineLearnt"

# labels: 1, clumps; 2, odd clumps; 3, stars; 4, galaxies; 5, bulge.

label_to_tool_map = {1:0, 2:0, 3:1, 4:2, 5:3}

This code generates the CSV file to be uploaded to Google Drive.

In [26]:
def filename_to_id(filename: str):
    pattern = filename.split("_")[1]
    if pattern.startswith("NEG"):
        return -int(pattern[3:])
    try:
        return int(pattern)
    except ValueError as e:
        print(e, pattern)
        return -1

## Build one Caesar upload table per manifest
Convert each subject manifest into Caesar extract rows by matching subject IDs back to predicted bounding boxes and serializing the resulting rectangle annotations as JSON payloads.

In [ ]:
def gen_caesar_upload_csv(subject_manifest):
    subject_manifest_df = pd.read_csv(subject_manifest).drop(columns="Unnamed: 0")

    anno_data = []

    for _, (subject_id, filename) in subject_manifest_df.iterrows():

        subject_local_id = filename_to_id(filename)
        file_anno_data = []

        file_payload = {
            "subject_id": subject_id,
            "extractor_key": extractor_key,
        }

        for label in unique_labels:
            if (subject_local_id, label) in bbox_groups.groups:
                bbg = bbox_groups.get_group((subject_local_id, label))
                for _, bbdata in bbg.iterrows():
                    width = 0.5 * (bbdata.px_x2 - bbdata.px_x1)
                    height = 0.5 * (bbdata.px_y2 - bbdata.px_y1)
                    if bbdata.scores > 0.2:
                        markId = "".join(
                            random.choice(
                                string.ascii_uppercase + string.ascii_lowercase + string.digits
                            )
                            for _ in range(6)
                        )

                        file_anno_data.append(
                            {
                                "stepKey": "S0",
                                "taskIndex": 0,
                                "taskKey": "T0",
                                "taskType": "drawing",
                                "toolIndex": label_to_tool_map[bbdata.labels],
                                "frame": 0,
                                "markId": markId,
                                "toolType": "rectangle",
                                "height": height,
                                "width": width,
                                "x_center": bbdata.px_centre_x,
                                "y_center": bbdata.px_centre_y,
                            }
                        )
        file_payload["data"] = json.dumps({"data": file_anno_data})

        anno_data.append(file_payload)

    anno_data_table = pd.DataFrame.from_records(anno_data)
    anno_data_table.to_csv(os.path.join(DATA_DIR, ANNOTATIONS_DIR, os.path.basename(subject_manifest).replace(".csv", "_annos.csv")), index=False)
    return anno_data_table

## Combine all generated annotation tables
Run the manifest-to-annotation conversion for every subject manifest and merge the results into a single CSV that can be uploaded to Google Drive for Caesar import.

In [28]:
all_caesar_upload_csvs = [gen_caesar_upload_csv(subject_manifest) for subject_manifest in subject_manifests]
pd.concat(all_caesar_upload_csvs).to_csv(os.path.join(DATA_DIR, ANNOTATIONS_DIR,"all_caesar_annotations.csv"), index=False)

## Part 2 - Import annotation data into Caesar database

First upload the file generated in Part I to Google drive and generate a sharable public link: 

e.g. https://drive.google.com/file/d/**1oAAlst10Nrt_sME4PT4ovdYT2bdmOfYe**/view?usp=share_link

The part of the link shown in bold above is the file's unique identifier. Your link will have a different identifier so copy it into the cell below as the value of the `file_id` variable.

In [ ]:
file_id = '<INSERT CODE>'
# workflow_id = '29070' # Correct a Clump
workflow_id = 32020 # testing

username = getpass.getpass("Panoptes username: ")
password = getpass.getpass("Panoptes password: ")

client = Panoptes.connect(username=username, password=password)

Panoptes.connect(username=username, password=password, endpoint='https://panoptes.zooniverse.org')
caesar = Caesar(endpoint='https://caesar.zooniverse.org')

res = caesar.import_data_extracts(workflow_id,f'https://drive.google.com/uc?export=download&id={file_id}')
print(res)

(None, None)


## Scratch cell
Leave this final cell empty for one-off checks, alternate file IDs, or retry commands while testing the Caesar import.